In [72]:
import importlib
import tenseal as ts
import utils

importlib.reload(utils)


<module 'utils' from 'f:\\PROJECTS\\research\\tenseal\\utils.py'>

## Generate Keys

In [73]:
context = ts.context(
    ts.SCHEME_TYPE.CKKS,
    poly_modulus_degree=8192,
    coeff_mod_bit_sizes = [60, 40, 40, 60]
)



In [74]:
context.generate_galois_keys()
context.global_scale = 2**40

In [75]:
secret_context = context.serialize(save_secret_key=True) #Serializes the full encryption setup including the secret key
utils.write_data("keys/secret.txt", secret_context) #saves it

In [76]:
context.make_context_public() #this drops the private key completely
public_context = context.serialize()
utils.write_data("keys/public.txt", public_context) 

## Encrypt

In [77]:
context = ts.context_from(utils.read_data("keys/secret.txt")) #since we dropped the private key before, we are reading it to get both the keys

In [78]:
salary = [10000]
bonus = [600]


In [79]:
salary_enc = ts.ckks_vector(context, salary) #encrypts the salary
bonus_enc = ts.ckks_vector(context, bonus) #encrypts the bonus

In [80]:
utils.write_data("outputs/salary_enc.txt", salary_enc.serialize())
utils.write_data("outputs/bonus_enc.txt", bonus_enc.serialize())

## Decryption after adding bonus (DON'T RUN IT BEFORE RUNNING THE OPERATOR)

In [82]:
m_proto = utils.read_data("outputs/salary_enc_new.txt") # Reads the serialized encrypted result from the file.
m = ts.lazy_ckks_vector_from(m_proto) # Reconstructs the encrypted CKKS vector without attaching the context yet.
m.link_context(context) # Links the TenSEAL context (keys + CKKS parameters
print(round(m.decrypt()[0], 2))

12600.0
